# Extract, Normalize and Chunk Example

In [1]:
# Colab dependency conflicts require a workaround installing dependencies. Don't ask...
!pip install "unstructured[pdf]" 
!pip install emoji psutil backoff dataclasses-json filetype langdetect python-iso639 python-magic python-oxmsg python-oxmsg rapidfuzz unstructured-client

!pip install pymupdf
!pip install -U langchain-community langchain-text-splitters

In [2]:
import textwrap as tw

## Load a PDF Document

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

# To get rid of the deprecation warning:
!pip install langchain_pymupdf4llm
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

C:\Users\Efrat\AppData\Local\Temp\ipykernel_29480\1913454369.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 3.4 MB/s eta 0:00:06
   -- ------------------------------------- 1.3/19.8 MB 3.8 MB/s eta 0:00:05
   ---- ----------------------------------- 2.4/19.8 MB 4.2 MB/s eta 0:00:05
   ------- -------------------------------- 3.7/19.8 MB 4.6 MB/s eta 0:00:04
   ---------- ----------------------------- 5.2/19.8 MB 5.2 MB/s eta 0:00:03
   ------------- -------------------------- 6.8/19.8 MB 5.6 MB/s eta 0:00:03
   ------------------ --------------------- 8.9/19.8 MB 6.2 MB/s eta 0:00:02
   ---------------------- ----------------- 11.3/19.8 MB 6.9 MB/s eta 0:00:02
   ---------------------------- ----------- 13.9/19.8 MB 7.5 MB/s eta 0:00:01
   --------------------------------- ------ 16.8/19.8 MB 8.2 MB/s eta 0:00:01
   ------------------------------------- -- 18.4/19.8 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 19.8/19.8 MB 8.1 MB/s  0:00:02
   -

In [4]:
loader = PyMuPDFLoader("exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf")
documents = loader.load()

In [5]:
documents[4].metadata

{'producer': '',
 'creator': '',
 'creationdate': '2025-03-23T17:49:51-04:00',
 'source': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'file_path': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'total_pages': 34,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-03-25T10:27:43-04:00',
 'trapped': '',
 'modDate': "D:20250325102743-04'00'",
 'creationDate': "D:20250323174951-04'00'",
 'page': 4}

In [6]:
documents[4].page_content

'[ 5 ] \nNONSTATE TRANSNATIONAL CRIMINALS \nAND TERRORISTS  \n \nTransnational criminals, terrorists, and other nonstate actors are threatening and impacting the lives of \nU.S. citizens, the security and prosperity of the Homeland, and U.S. strength at home and abroad. Some \ntransnational criminal organizations (TCOs) are producing and trafficking large amounts of illicit drugs \nthat are imperiling American lives and livelihoods. They are conducting other illegal activities that \nchallenge U.S. security, such as human trafficking, cyber operations, money laundering, and inciting \nviolence. U.S. citizens—at home and abroad—are also facing more diverse, complex, and decentralized \nterrorist threats. Actors, ranging from designated Foreign Terrorist Organizations—including the Islamic \nState of Iraq and ash-Sham (ISIS), al-Qa‘ida, other Islamist terrorist groups, and some drug cartels— to \nterrorists acting alone or in small cells, are likely to pursue, enable, or inspire attacks.

## Cleanup and Normalize

In [7]:
from unstructured.cleaners.core import clean, clean_bullets, clean_extra_whitespace

def clean_text(text: str) -> str:
    
    text = clean(text)
    text = clean_bullets(text)
    text = clean_extra_whitespace(text)
    
    return text.strip()

In [9]:
for doc in documents:
    doc.page_content = clean_text(doc.page_content)

In [10]:
documents[4].page_content

'[ 5 ] NONSTATE TRANSNATIONAL CRIMINALS AND TERRORISTS Transnational criminals, terrorists, and other nonstate actors are threatening and impacting the lives of U.S. citizens, the security and prosperity of the Homeland, and U.S. strength at home and abroad. Some transnational criminal organizations (TCOs) are producing and trafficking large amounts of illicit drugs that are imperiling American lives and livelihoods. They are conducting other illegal activities that challenge U.S. security, such as human trafficking, cyber operations, money laundering, and inciting violence. U.S. citizens—at home and abroad—are also facing more diverse, complex, and decentralized terrorist threats. Actors, ranging from designated Foreign Terrorist Organizations—including the Islamic State of Iraq and ash-Sham (ISIS), al-Qa‘ida, other Islamist terrorist groups, and some drug cartels— to terrorists acting alone or in small cells, are likely to pursue, enable, or inspire attacks. Finally, large-scale ille

## Chunking

#### Character-Based Recursive Chunking

The LangChain splitter class is named `RecursiveCharacterTextSplitter`. See **[here](https://gauravbytes.dev/chunking-strategies-in-langchain-rag-a-deep-dive)** to learn how it works.

The `chunk_size` and `chunk_overlap` arguments are in **characters**, not words or tokens. The average length of a word in English is 4.7 characters, and the average length of a token is $4.7 / 1.4 \sim 3.3$ characters. Therefore, if our context window is, say, $384$ tokens, then we can have chunks up to $~1200$ characters long (+ a safety margin).

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = splitter.split_documents(documents[3:])

In [13]:
len(chunks)

95

In [14]:
chunks[0].metadata

{'producer': '',
 'creator': '',
 'creationdate': '2025-03-23T17:49:51-04:00',
 'source': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'file_path': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'total_pages': 34,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-03-25T10:27:43-04:00',
 'trapped': '',
 'modDate': "D:20250325102743-04'00'",
 'creationDate': "D:20250323174951-04'00'",
 'page': 3}

In [15]:
chunks[1].metadata

{'producer': '',
 'creator': '',
 'creationdate': '2025-03-23T17:49:51-04:00',
 'source': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'file_path': 'exercises/assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf',
 'total_pages': 34,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-03-25T10:27:43-04:00',
 'trapped': '',
 'modDate': "D:20250325102743-04'00'",
 'creationDate': "D:20250323174951-04'00'",
 'page': 3}

In [16]:
len(chunks[0].page_content)

1194

In [17]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

[ 4 ] FOREWORD The 2025 Annual Threat Assessment (ATA) is the
Intelligence Community’s (IC) official, coordinated evaluation of an
array of threats to U.S. citizens, the Homeland, and U.S. interests in
the world. A diverse set of foreign actors are targeting U.S. health
and safety, critical infrastructure, industries, wealth, and
government. State adversaries and their proxies are also trying to
weaken and displace U.S. economic and military power in their regions
and across the globe. Both state and nonstate actors pose multiple
immediate threats to the Homeland and U.S. national interests.
Terrorist and transnational criminal organizations are directly
threatening our citizens. Cartels are largely responsible for the more
than 55,000 U.S. deaths from synthetic opioids in the 12 months ending
in September 2024 and helped facilitate the nearly three million
illegal migrant arrivals at the Southwest Border in 2024, straining
resources and putting U.S. communities at risk. A range of cyb

#### Sentence-Based Chunking

In [18]:
!python -m spacy download en

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.8 MB 5.2 MB/s eta 0:00:03
     ------------ --------------------------- 3.9/12.8 MB 8.0 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 10.0 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 10.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 11.4 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [19]:
from langchain_text_splitters import SpacyTextSplitter

In [20]:
splitter = SpacyTextSplitter(chunk_size=1200)  
chunks = splitter.split_documents(documents[3:])

c:\Users\Efrat\Downloads\KodCode-AI-Augmented-Software-Development\.venv\Lib\site-packages\spacy\pipeline\lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [21]:
len(chunks)

94

In [22]:
len(chunks[0].page_content)

1103

In [23]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

[ 4 ] FOREWORD The 2025 Annual Threat Assessment (ATA) is the
Intelligence Community’s (IC) official, coordinated evaluation of an
array of threats to U.S. citizens, the Homeland, and U.S. interests in
the world.  A diverse set of foreign actors are targeting U.S. health
and safety, critical infrastructure, industries, wealth, and
government.  State adversaries and their proxies are also trying to
weaken and displace U.S. economic and military power in their regions
and across the globe.  Both state and nonstate actors pose multiple
immediate threats to the Homeland and U.S. national interests.
Terrorist and transnational criminal organizations are directly
threatening our citizens.  Cartels are largely responsible for the
more than 55,000 U.S. deaths from synthetic opioids in the 12 months
ending in September 2024 and helped facilitate the nearly three
million illegal migrant arrivals at the Southwest Border in 2024,
straining resources and putting U.S. communities at risk.  A range o

#### Semantic Chunking

Steps:
1. The document is first broken down into individual sentences.
2. An embedding model converts each sentence into a numerical vector representation.
3. Semantic similarity is calculated between consecutive sentence pairs.
4. Wherever the similarity score between two consecutive sentences drops significantly, the algorithm infers a shift in topic and places a chunk boundary.

LangChain's variant: "At a high level, this splits into sentences, then groups into groups of 3 sentences, and then merges one that are similar in the embedding space." (LangChain documentation)

In [24]:
!pip install langchain-experimental langchain_huggingface

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

C:\Users\Efrat\AppData\Local\Temp\ipykernel_29480\2950647526.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [26]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = SemanticChunker(
    embeddings=embedding_model,
    # See: https://python.langchain.com/docs/how_to/semantic-chunker/#breakpoints
    breakpoint_threshold_type="percentile",   # 'standard_deviation', 'interquartile', 'gradient'
    breakpoint_threshold_amount=90.0,
)

chunks = splitter.split_documents(documents[3:])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
len(chunks)

In [ ]:
len(chunks[0].page_content)

In [ ]:
print(tw.fill(chunks[0].page_content))
print("~" * 100)
print(tw.fill(chunks[1].page_content))
print("~" * 100)
print(tw.fill(chunks[2].page_content))

<hr/>